[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-01.ipynb)

# 1주차 실습: 시퀀스 길이와 어텐션 비용

- 목표: 제공된 PyTorch 어텐션을 CPU에서 실행하고 길이만 바꾸어 점수표 크기·연산량·시간 비교
- 변경 지점: 기본 시퀀스 길이 한 값. 변경 전에도 위에서부터 끝까지 실행 가능
- 데이터: 위키백과 「제주특별자치도」 고정 판 42464087, CC BY-SA 4.0
- 원문·저자 이력: https://ko.wikipedia.org/w/index.php?oldid=42464087
- 라이선스: https://creativecommons.org/licenses/by-sa/4.0/
- 문자 단위 교육 입력. 실제 언어모델 토크나이저·학습 모델·GPU 지연 측정 아님
- 공개 문서 다운로드를 위한 인터넷 필요. 원문 접근 실패를 작성 문장으로 대체하지 않는 원칙


## 0. 준비

- 무료 CPU 런타임 사용, 전용 GPU 불필요
- PyTorch가 이미 있으면 현재 버전 유지. 없을 때만 호환 범위의 라이브러리 설치
- 비교표와 함께 실행 환경·라이브러리 버전 기록


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch>=2.2,<3"])

import torch
import platform
import time
import statistics

# CPU 비교에서 스레드 수 고정. GPU를 사용하지 않는 교육 예제
torch.set_num_threads(2)
torch.manual_seed(2026)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("장치: CPU, dtype: float32, 스레드:", torch.get_num_threads())


## 1. 제주 공개 문서 불러오기

- 동일 판의 본문 문단을 다운로드하고 문자 목록으로 변환
- 문서 길이보다 긴 입력은 같은 문자를 순환 반복해 구성. 새 제주 사실을 생성하는 과정이 아닌 계산 크기 비교용 반복
- 첫 문단 미리보기와 판 번호로 입력 출처 확인


In [ ]:
import urllib.request
import urllib.parse
import urllib.error
import json
import re
import html

OLDID = 42464087
SOURCE = f"https://ko.wikipedia.org/w/index.php?oldid={OLDID}"
url = "https://ko.wikipedia.org/w/api.php?" + urllib.parse.urlencode({
    "action": "parse", "oldid": OLDID, "prop": "text", "format": "json", "formatversion": 2})
request = urllib.request.Request(url, headers={"User-Agent": "deepnlp-course/1.0 (yj.lee@chu.ac.kr)"})
# 일시 제한에는 한 차례만 재시도. 원문을 얻지 못하면 출처 오류를 숨기지 않음
for attempt in range(2):
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            payload = json.load(response)
        break
    except (urllib.error.URLError, TimeoutError):
        if attempt == 1:
            raise RuntimeError("제주 고정 판 다운로드 실패. 인터넷·원문 접근을 확인한 후 이 셀 재실행.")
        time.sleep(2)
if "parse" not in payload:
    raise RuntimeError(f"원문 API 응답 확인 필요: {payload.get('error', 'parse 없음')}")
raw = payload["parse"]["text"]
def clean(paragraph):
    paragraph = re.sub(r"<sup.*?</sup>", "", paragraph, flags=re.S)
    paragraph = re.sub(r"<[^>]+>", "", paragraph)
    return re.sub(r"\s+", " ", html.unescape(paragraph)).strip()
paragraphs = [clean(p) for p in re.findall(r"<p>(.*?)</p>", raw, flags=re.S)]
paragraphs = [p for p in paragraphs if len(p) > 40]
text = " ".join(paragraphs)
if not text:
    raise RuntimeError("다운로드했지만 본문 문단 없음. API 응답과 문서 구조 확인 필요.")
characters = sorted(set(text))
char_to_id = {char: index for index, char in enumerate(characters)}
source_ids = torch.tensor([char_to_id[char] for char in text], dtype=torch.long)
print("출처:", SOURCE)
print("라이선스: CC BY-SA 4.0")
print("본문 문자 수:", len(text), "/ 문자 종류:", len(characters))
print("본문 미리보기:", text[:160])


## 2. 먼저 제공된 어텐션 실행

- 임베딩과 Q·K·V 투영은 seed를 고정한 미학습 가중치. 결과의 언어 의미·검색 품질을 평가하지 않는 실습
- `n`: 문자 위치 수, `d = 32`: 각 벡터의 차원
- 점수 `S = Q @ K.T / sqrt(d)`, 비율 `A = softmax(S)`, 출력 `Y = A @ V`
- 점수와 비율 모두 `n × n`, 출력은 `n × d`
- 제공 함수에서 고칠 부분 없음


In [ ]:
D = 32
embedding = torch.randn(len(characters), D, dtype=torch.float32) / (D ** 0.5)
w_q = torch.randn(D, D, dtype=torch.float32) / (D ** 0.5)
w_k = torch.randn(D, D, dtype=torch.float32) / (D ** 0.5)
w_v = torch.randn(D, D, dtype=torch.float32) / (D ** 0.5)

def make_qkv(n):
    # 공개 문자의 순환 반복. 자료의 의미·길이 분포 재현을 주장하지 않음
    repeats = (n + len(source_ids) - 1) // len(source_ids)
    ids = source_ids.repeat(repeats)[:n]
    x = embedding[ids]
    return x @ w_q, x @ w_k, x @ w_v

def attention(q, k, v):
    scores = (q @ k.T) / (D ** 0.5)
    weights = torch.softmax(scores, dim=-1)
    output = weights @ v
    return scores, weights, output

with torch.inference_mode():
    q, k, v = make_qkv(4)
    scores, weights, output = attention(q, k, v)
print("입력 문자:", list(text[:4]))
print("Q·K·V 모양:", tuple(q.shape))
print("점수표 모양:", tuple(scores.shape))
print("점수표:", scores)
print("비율표:", weights)
print("각 행 합:", weights.sum(dim=-1))
print("출력 모양:", tuple(output.shape))
assert torch.allclose(weights.sum(dim=-1), torch.ones(4), atol=1e-6)


## 3. 시퀀스 길이 한 곳만 변경

- 기본값 64에서 `[64, 128, 256]` 비교표 생성
- 다음 실행에서 128로 바꾸면 `[128, 256, 512]` 비교 가능
- 허용 범위 8~512. 가장 긴 비교 입력은 기본값의 4배, CPU와 무료 런타임 규모 제한


In [ ]:
BASE_LENGTH = 64  # TODO: 이 시퀀스 길이 값 한 곳만 8~512 정수로 변경
if not isinstance(BASE_LENGTH, int) or not 8 <= BASE_LENGTH <= 512:
    raise ValueError("BASE_LENGTH는 8~512 정수로 설정.")
lengths = [BASE_LENGTH, 2 * BASE_LENGTH, 4 * BASE_LENGTH]
print("비교할 문자 위치 수:", lengths)


## 4. 메모리·연산량·CPU 시간 비교표

- 점수표 이론 MiB: `n² × 4 / 1024²`. float32 한 칸 4바이트, batch 1·head 1·점수표 한 장 기준
- 점수표 텐서의 실제 저장 바이트: `scores.numel() × scores.element_size()`로 별도 확인
- 연산량 MAC: QK 곱과 AV 곱만 `2 × n² × d`. MAC은 곱셈 후 누적 한 쌍의 계산 단위
- 투영, softmax, 나눗셈, 자료 로딩을 제외한 MAC 추정. 실제 CPU 명령 수 아님
- CPU 시간: Q·K·V를 준비한 뒤 제공 어텐션 함수 전체(점수·softmax·가중합)를 준비 실행 1회, 측정 3회 후 중앙값
- 측정 범위가 다른 MAC과 시간을 비례 관계의 실측 증거로 곧바로 비교하지 않는 주의
- **peak 메모리 미측정**. softmax 비율표·출력·QKV·라이브러리 작업 공간 등을 포함한 프로세스 전체 메모리는 점수표 한 장보다 큰 값

- 비용 열: 측정 CPU 초에 작성 가정 단가 `0.02원/초`를 곱한 시나리오 계산. 외부 가격·실제 청구액 아님. 자료 로딩·투영·다른 모델 연산·사람 검토는 비용 계산에서 제외


In [ ]:
# 비용 교육 시나리오: 초당 0.02원으로 가정. 실제 서비스·컴퓨팅 가격 조사 아님
ASSUMED_WON_PER_CPU_SECOND = 0.02
rows = []
with torch.inference_mode():
    for n in lengths:
        q, k, v = make_qkv(n)
        # 준비 실행. 이전 실행의 반환 텐서를 쌓아 두지 않음
        warmup = attention(q, k, v)
        del warmup
        timings = []
        tensor_bytes = 0
        for repeat in range(3):
            started = time.perf_counter()
            measured_scores, measured_weights, measured_output = attention(q, k, v)
            timings.append((time.perf_counter() - started) * 1000)
            tensor_bytes = measured_scores.numel() * measured_scores.element_size()
            assert measured_scores.shape == (n, n)
            assert measured_output.shape == (n, D)
            del measured_scores, measured_weights, measured_output
        expected_bytes = n * n * 4
        assert tensor_bytes == expected_bytes
        rows.append({"n": n, "cells": n*n, "theory_bytes": expected_bytes,
                     "tensor_bytes": tensor_bytes, "mac": 2*n*n*D,
                     "median_ms": statistics.median(timings)})
        del q, k, v

print("n | 점수표칸 | 이론MiB | 저장MiB | QK+AV MAC | CPU 중앙값ms | 가정비용원 | peak메모리")
for row in rows:
    print(f"{row['n']} | {row['cells']} | {row['theory_bytes']/1024**2:.4f} | "
          f"{row['tensor_bytes']/1024**2:.4f} | {row['mac']} | {row['median_ms']:.4f} | "
          f"{row['median_ms']/1000*ASSUMED_WON_PER_CPU_SECOND:.8f} | 미측정")
print("가정비용 = 측정 CPU 초 × 0.02원/초. 작성 가정이며 실제 가격·전체 요청 비용 아님.")
print("\n첫 행 대비 이론 점수표·MAC 비율:")
for row in rows:
    print(f"n={row['n']}: {row['cells']/rows[0]['cells']:.1f}배")


## 5. 결과 해석과 제출 연습

- `n`이 2배면 점수표와 위 MAC은 4배, 4배면 16배. 이론·텐서 저장 바이트는 일치하는 항목
- CPU 중앙값이 정확히 4배·16배가 아니어도 오류로 확정 불가. 작은 입력의 호출 부담, CPU 상태, 캐시, 연산 구현의 영향
- 실제 언어모델은 여러 층·head·batch·투영·KV 캐시를 포함. 이 점수표 한 장의 메모리를 모델 전체 요구 메모리로 쓰지 않는 해석
- 가정 제약 예: 점수표 한 장 1MiB 이내, 제공 CPU 함수 중앙값 10ms 이내. 작성 시나리오이며 서비스 전체 메모리·지연 예산을 충족했다는 판단으로 확대 금지
- 이 실습은 forward 계산. 학습용 gradient·optimizer 메모리와 GPU latency·실제 원화 비용 미측정. 가정비용 열은 초당 0.02원의 작성 시나리오

| 상정 조직의 조건 | 실제 채울 근거 | 아직 모르는 항목 | 가능한 판단 |
|---|---|---|---|
| 전용 GPU 없음, 짧은 문의 위주 | CPU 표의 길이·실행 조건·시간 | 실제 요청 분포·동시 사용자 | 측정한 작은 입력에서의 실행 가능 범위 |
| 긴 문서를 통째로 입력 | 길이 증가의 점수표·MAC 비율 | 전체 모델 peak·서비스 지연 | 문서 길이를 줄이거나 검색 결합을 검토할 필요 |

제출 연습용 양식(새 배점·마감 지정 없음):

```text
내 학번:
실행 환경·Python·PyTorch·CPU 스레드:
기본 길이와 바꾼 길이:
출력 비교표:
점수표·MAC의 증가 이유:
CPU 시간이 이론 비율과 다른 이유 후보:
도내 소규모 조직의 상정 메모리·시간 제약:
실측한 시간·텐서 저장 바이트 / 이론 MAC / 가정 단가 비용 / 미측정 peak·GPU·실제 비용:
이 표로 판단할 수 있는 범위:
```

- 수업의 제출 절차를 연습할 때 저장 경로: `assignments/week-01/<학번>/`
- 실행한 노트북과 해석 표만 준비. 학생 폴더에 이름·연락처 기재 금지
- 실제 제출 여부·방법·기한은 수업 공지와 강의 사이트의 과제 제출 안내 확인
- 모르는 개념 한 마디를 1주차 사전 진단에 추가, 설명 가능한 근거 한 가지와 남은 질문 기록

## 필요한 출처

- 위키백과 「제주특별자치도」 고정 판과 저자 목록: https://ko.wikipedia.org/w/index.php?oldid=42464087
- CC BY-SA 4.0: https://creativecommons.org/licenses/by-sa/4.0/
- PyTorch softmax 공식 문서: https://docs.pytorch.org/docs/stable/generated/torch.softmax.html
- 이번 코드는 학습되지 않은 단일 head, batch 1의 교육용 어텐션. 실제 제주 서비스 품질을 측정한 결과 아님
